# Run 5-hop pipeline with Ollama on Google Colab

**Before running:**
1. Upload your project to Colab: **File → Upload notebook** (this file), then **File → Upload** the rest of the repo as a zip, or clone from GitHub (see cell below).
2. Run cells in order. The first time will install Ollama and pull a small model (~1–2 min).
3. Use a **small model** (e.g. `gemma3:1b` or `llama3.2:1b`) to stay within Colab RAM (~12GB free tier).

## 1. (Optional) Clone repo from GitHub

If your project is on GitHub, run this once. Otherwise skip and use your uploaded folder.

In [ ]:
# Uncomment and set your repo URL, then run this cell.
# !git clone https://github.com/YOUR_USER/uoa-group1-c6.git /content/uoa-group1-c6
# %cd /content/uoa-group1-c6

## 2. Install Ollama (Linux) and start server

In [ ]:
import subprocess
import time
import shutil

# Install Ollama if not present
if not shutil.which("ollama"):
    print("Installing Ollama...")
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
else:
    print("Ollama already installed.")

# Start Ollama server in background (skip if already running)
try:
    subprocess.Popen(
        ["ollama", "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    time.sleep(3)
    print("Ollama server started.")
except Exception as e:
    print("Ollama serve (may already be running):", e)

## 3. Pull a small model (run once per session)

In [ ]:
# Use a small model to fit Colab RAM (~12GB). Options: gemma3:1b, llama3.2:1b, phi3:mini
OLLAMA_MODEL = "gemma3:1b"

!ollama pull {OLLAMA_MODEL}

## 4. Install Python dependencies

In [ ]:
from pathlib import Path

# Project root (parent of notebooks/)
ROOT = Path.cwd() if "uoa-group1-c6" in str(Path.cwd()) else Path.cwd().parent
if not (ROOT / "requirements.txt").exists():
    ROOT = Path("/content/uoa-group1-c6")  # Colab default after clone

%cd {ROOT}
!pip install -q -r requirements.txt
print("Requirements installed. Project root:", ROOT)

## 5. Imports and 5-hop pipeline setup

In [ ]:
import sys
import os
from pathlib import Path

import pandas as pd

# Ensure project root on path (Colab: often /content/uoa-group1-c6)
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
if not (ROOT / "src").exists():
    ROOT = Path("/content/uoa-group1-c6")
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

# Optional: set Ollama env (defaults: localhost:11434)
os.environ.setdefault("OLLAMA_HOST", "http://localhost:11434")
os.environ.setdefault("OLLAMA_MODEL", "gemma3:1b")

from src.pipeline.llm_client import LLMClient
from src.pipeline.orchestrator import ReasoningPipeline

llm = LLMClient(max_tokens=128, model=os.environ["OLLAMA_MODEL"])
pipeline = ReasoningPipeline(llm_client=llm)
print("LLMClient and ReasoningPipeline ready.")

## 6. Load data and run pipeline on a sample

In [ ]:
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
if not (ROOT / "src").exists():
    ROOT = Path("/content/uoa-group1-c6")

csv_path = ROOT / "sentiment_predictions_allday_articles.csv"
if not csv_path.exists():
    raise FileNotFoundError(f"Data not found: {csv_path}. Upload the CSV or clone the repo.")

df = pd.read_csv(csv_path)
sample_df = df.sample(n=min(10, len(df)), random_state=42).reset_index(drop=True)

results = []
for _, row in sample_df.iterrows():
    try:
        ctx = pipeline.run(text=row["title"], ticker=row["ticker"])
        final = pipeline.get_final_result(ctx)
    except Exception as e:
        final = {"error": str(e)}
    results.append(final)

sample_df["hop_sentiment"] = [r.get("sentiment") for r in results]
sample_df["hop_sentiment_score"] = [
    ({"Positive": 1, "Negative": -1}.get(str(r.get("sentiment")).strip(), 0) for r in results
]
sample_df[["title", "ticker", "hop_sentiment", "hop_sentiment_score"]].head()

## 7. (Optional) Evaluate vs GPT reference column

In [ ]:
from src.evaluation.metrics import compute_classification_metrics

y_pred = sample_df["hop_sentiment_score"]
y_true = sample_df["gpt_sentiment_p6"]
metrics = compute_classification_metrics(y_true, y_pred)
print("Accuracy:", metrics.get("accuracy"))
print("F1 (macro):", metrics.get("f1_macro"))
print("Per-class:", metrics.get("per_class"))